# Child population by floor-area band (superseded)

**What this was for.** The first method for putting children aged 0–9 onto HDB blocks. SingStat's
`respopagesexfa2026` table gives residents by subzone, age and **floor-area band** (≤60, 60–80,
80–100, 100–120, >120 m²), for all housing. Each block's flats were mapped to a band using the
median floor area of that block's resale transactions (five HDB Resale Flat Prices datasets,
1990–present, downloaded from data.gov.sg), with a national median as fallback for rental and
studio units that never appear in resale data; children in each subzone × band were then spread
over the HDB units in that band. Because the population table covers private housing too, roughly
a fifth of the children stay unassigned by design.

**Why it was superseded** (decision 2026-10-22, `docs/decisions.md`; details in
`output/hdb_unresolved_blocks_handling.md`). The type-of-dwelling table lets children be split by
flat type directly, with no floor-area estimate, no resale download and an exact HDB-only total,
so [`../1_Data_Preparation/1_04_assign_child_population.ipynb`](../1_Data_Preparation/1_04_assign_child_population.ipynb)
(Jessie) is the canonical method.

**Not re-executed.** The notebook downloads five resale datasets (~1 million rows) at run time and
reads `respopagesexfa2026.csv` from outside the repository (`POP_CSV`, default
`~/Downloads/respopagesexfa2026/respopagesexfa2026.csv`). Its last output,
`output/hdb_blocks_with_children.gpkg` (10,741 blocks, the pre-fallback block list), is kept
unchanged for the record. Original script: `scripts/assign_child_population.py` (Kenneth).

**Inputs**
- `output/hdb_blocks.gpkg`, layer `hdb_blocks` (1_02).
- `data/MasterPlanSubZones_2019.geojson`.
- `respopagesexfa2026.csv` (SingStat, outside the repo) and the five resale datasets (downloaded).

**Outputs**
- `output/hdb_blocks_with_children.gpkg`, layer `hdb_blocks_with_children` — `child_pop_0_9` and
  `units_<band>` per residential block.

In [ ]:
import sys
from pathlib import Path

import io
import os
import time

import pandas as pd
import geopandas as gpd
import requests

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

# changed 2026-11-10: paths through bt4015.paths; POP_CSV still comes from the environment or ~/Downloads.
POP_CSV = Path(
    os.environ.get(
        "POP_CSV",
        Path.home() / "Downloads" / "respopagesexfa2026" / "respopagesexfa2026.csv",
    )
)
HDB_GPKG = OUT / "hdb_blocks.gpkg"
SUBZONE_GEOJSON = DATA / "MasterPlanSubZones_2019.geojson"
OUT_GPKG = OUT / "hdb_blocks_with_children.gpkg"
OUT_LAYER = "hdb_blocks_with_children"

RESALE_DATASET_IDS = [
    "d_ea9ed51da2787afaf8e51f827c304208",   # 1990-1999
    "d_ebc5ab87086db484f88045b47411ebc5",   # 2000-Feb 2012
    "d_2d5ff9ea31397b66239f245f57751537",   # Mar 2012-Dec 2014
    "d_43f493c6c50d54243cc1eab0df142d6a",   # Jan 2015-Dec 2016
    "d_8b84c4ee58e3cfc0ece0d773c8ca6abc",   # Jan 2017-present
]
FA_BUCKETS = ["<=60", ">60 to 80", ">80 to 100", ">100 to 120", ">120"]   # SingStat floor-area bands

# Same abbreviation table as 1_02, so resale street names normalise to the same join_key.
ABBR = {
    "ROAD": "RD", "STREET": "ST", "AVENUE": "AVE", "DRIVE": "DR",
    "CRESCENT": "CRES", "JALAN": "JLN", "LORONG": "LOR", "NORTH": "NTH",
    "SOUTH": "STH", "CENTRAL": "CTRL", "PLACE": "PL", "TERRACE": "TER",
    "GARDENS": "GDNS", "GARDEN": "GDN", "PARK": "PK", "CLOSE": "CL",
    "HEIGHTS": "HTS", "GROVE": "GR", "BOULEVARD": "BLVD", "HIGHWAY": "HWY",
    "UPPER": "UPP", "BUKIT": "BT", "KAMPONG": "KG", "TANJONG": "TG",
    "TAMAN": "TMN", "GREEN": "GRN", "COMMONWEALTH": "C'WEALTH", "SQUARE": "SQ",
}
# HDB Property Information column -> resale flat_type label. Rental and studio units never appear
# in resale data; they fall back to the national median of the closest equivalent type.
PROP_COL_TO_FLAT_TYPE = {
    "1room_sold": "1 ROOM", "2room_sold": "2 ROOM", "3room_sold": "3 ROOM",
    "4room_sold": "4 ROOM", "5room_sold": "5 ROOM", "exec_sold": "EXECUTIVE",
    "multigen_sold": "MULTI-GENERATION",
    "studio_apartment_sold": "1 ROOM",      # ~35 sqm, falls in <=60 bucket
    "1room_rental": "1 ROOM", "2room_rental": "2 ROOM", "3room_rental": "3 ROOM",
    "other_room_rental": "1 ROOM",
}

## 1. Helpers: floor-area band, join key, resale download

In [ ]:
def assign_fa_bucket(sqm: float) -> str:
    """Map a floor area value (sqm) to the FA bucket label used in the population CSV."""
    if sqm <= 60:
        return "<=60"
    elif sqm <= 80:
        return ">60 to 80"
    elif sqm <= 100:
        return ">80 to 100"
    elif sqm <= 120:
        return ">100 to 120"
    else:
        return ">120"


def normalize_street(name: str) -> str:
    """Uppercase, strip, then apply HDB abbreviations (matches 1_02)."""
    return " ".join(ABBR.get(w, w) for w in str(name).strip().upper().split())


def make_join_key(block: str, street: str) -> str:
    return f"{str(block).strip().upper()}_{normalize_street(street)}"


def download_resale_datasets() -> pd.DataFrame:
    """Download all 5 resale datasets; return concatenated DataFrame with
    block, street_name, flat_type, and floor_area_sqm columns."""
    api = "https://api-open.data.gov.sg/v1/public/api/datasets/{}/poll-download"
    frames = []
    for i, ds_id in enumerate(RESALE_DATASET_IDS):
        if i > 0:
            time.sleep(3)   # avoid 429 rate-limit
        print(f"  Fetching {ds_id} ...")
        meta = requests.get(api.format(ds_id), timeout=30)
        meta.raise_for_status()
        download_url = meta.json()["data"]["url"]
        csv_r = requests.get(download_url, timeout=180)
        csv_r.raise_for_status()
        df = pd.read_csv(
            io.StringIO(csv_r.text),
            usecols=["block", "street_name", "flat_type", "floor_area_sqm"],
        )
        frames.append(df)
        print(f"    {len(df):,} rows")
    return pd.concat(frames, ignore_index=True)

## 2. Floor-area lookups from resale transactions

Per (block, flat type) the median floor area of all resale transactions; nationally per flat type
as the fallback for units with no resale history.

In [ ]:
def build_fa_lookups(resale_df: pd.DataFrame):
    """
    Returns:
      block_lookup  - dict {(join_key, flat_type): fa_bucket}  per-block lookup
      national_map  - dict {flat_type: fa_bucket}              national-median fallback
    """
    resale_df = resale_df.copy()
    resale_df["flat_type"] = resale_df["flat_type"].str.strip().str.upper()
    resale_df["join_key"] = resale_df.apply(
        lambda r: make_join_key(r["block"], r["street_name"]), axis=1
    )

    # 1. Per-block median floor area per flat type.
    per_block = (
        resale_df.groupby(["join_key", "flat_type"])["floor_area_sqm"]
        .median()
        .reset_index()
    )
    per_block["fa_bucket"] = per_block["floor_area_sqm"].apply(assign_fa_bucket)
    block_lookup = {
        (row.join_key, row.flat_type): row.fa_bucket
        for row in per_block.itertuples()
    }

    # 2. National median fallback.
    national = (
        resale_df.groupby("flat_type")["floor_area_sqm"]
        .median()
        .reset_index()
    )
    national["fa_bucket"] = national["floor_area_sqm"].apply(assign_fa_bucket)
    national_map = national.set_index("flat_type")["fa_bucket"].to_dict()

    # 3. Handle MULTI GENERATION / MULTI-GENERATION naming inconsistency.
    for key in list(national_map.keys()):
        if "MULTI" in key:
            national_map["MULTI-GENERATION"] = national_map[key]
            national_map["MULTI GENERATION"] = national_map[key]

    print("\nNational fallback - flat type -> FA bucket:")
    print(national[["flat_type", "floor_area_sqm", "fa_bucket"]].to_string(index=False))
    print(f"\nPer-block lookup entries: {len(block_lookup):,}")
    return block_lookup, national_map


print("Downloading HDB Resale Flat Prices datasets ...")
resale_df = download_resale_datasets()
block_lookup, national_map = build_fa_lookups(resale_df)

## 3. Residential blocks and their subzone

In [ ]:
# 1. Residential blocks from the join (note: this method used residential == "Y", not is_residential).
hdb = gpd.read_file(HDB_GPKG, layer="hdb_blocks")
hdb = hdb[hdb["residential"] == "Y"].copy()
print(f"  {len(hdb):,} residential blocks")

# 2. Block centroid -> subzone.
subzones = gpd.read_file(SUBZONE_GEOJSON)[["SUBZONE_N", "geometry"]].to_crs(hdb.crs)
centroids = hdb[["geometry"]].copy()
centroids["geometry"] = hdb.centroid
joined = gpd.sjoin(centroids, subzones, how="left", predicate="within")
hdb["subzone"] = joined["SUBZONE_N"].values
unmatched = hdb["subzone"].isna().sum()
print(f"  Matched {len(hdb) - unmatched:,} blocks; {unmatched} unmatched (excluded)")

## 4. Units per floor-area band in each block

In [ ]:
def add_units_per_bucket(hdb, block_lookup, national_map):
    """
    For each block and each flat-type column, look up the actual FA bucket
    for that specific block from resale data. Falls back to the national median
    if the block+flat_type combination has no resale history.
    Adds units_<bucket> columns to hdb. Returns (hdb, stats_dict).
    """
    for b in FA_BUCKETS:
        hdb[f"units_{b}"] = 0.0

    per_block_units = 0.0
    fallback_units = 0.0

    for prop_col, flat_type in PROP_COL_TO_FLAT_TYPE.items():
        if prop_col not in hdb.columns:
            continue

        buckets = []
        for jk in hdb["join_key"]:
            b = block_lookup.get((jk, flat_type))
            if b is None:
                b = national_map.get(flat_type)
                is_fallback = True
            else:
                is_fallback = False
            buckets.append((b, is_fallback))

        bucket_series = pd.Series([x[0] for x in buckets], index=hdb.index)
        fallback_mask = pd.Series([x[1] for x in buckets], index=hdb.index)
        units = hdb[prop_col].fillna(0)

        per_block_units += units[~fallback_mask].sum()
        fallback_units += units[fallback_mask].sum()

        for bucket in FA_BUCKETS:
            mask = bucket_series == bucket
            hdb.loc[mask, f"units_{bucket}"] += units[mask]

    total_units = per_block_units + fallback_units
    stats = {
        "total_units": total_units,
        "per_block_units": per_block_units,
        "fallback_units": fallback_units,
    }
    return hdb, stats


print("Assigning flat types to FA buckets (per-block) ...")
hdb, unit_stats = add_units_per_bucket(hdb, block_lookup, national_map)

## 5. Children by subzone and band, and the dasymetric split

The population file covers all residents, so children in private housing remain unassigned; that
is expected (about 78% of residents live in HDB flats).

In [ ]:
def load_child_pop(pop_csv: Path) -> pd.DataFrame:
    """Load respopagesexfa2026.csv; return child pop (0-9) by subzone x FA bucket."""
    df = pd.read_csv(pop_csv)
    child = df[df["AG"].isin(["0_to_4", "5_to_9"])].copy()
    agg = (
        child.groupby(["SZ", "FA"])["Pop"]
        .sum()
        .reset_index()
        .rename(columns={"SZ": "subzone", "FA": "fa_bucket", "Pop": "child_pop"})
    )
    agg["subzone"] = agg["subzone"].str.upper()
    return agg


print(f"Loading child population from {POP_CSV} ...")
child_pop = load_child_pop(POP_CSV)

# 1. Within each subzone x band, children are split over blocks by their units in that band.
print("Disaggregating child population to blocks ...")
hdb["child_pop_0_9"] = 0.0
for fa_bucket in FA_BUCKETS:
    units_col = f"units_{fa_bucket}"
    sz_totals = (
        hdb.groupby("subzone")[units_col]
        .sum()
        .rename("sz_total_units")
        .reset_index()
    )
    fa_pop = child_pop[child_pop["fa_bucket"] == fa_bucket][["subzone", "child_pop"]]
    tmp = (
        hdb[["subzone", units_col]]
        .merge(sz_totals, on="subzone", how="left")
        .merge(fa_pop, on="subzone", how="left")
    )
    tmp["child_pop"] = tmp["child_pop"].fillna(0)
    tmp["sz_total_units"] = tmp["sz_total_units"].replace(0, pd.NA)
    hdb["child_pop_0_9"] += (
        (tmp[units_col] / tmp["sz_total_units"] * tmp["child_pop"]).fillna(0)
    )

assigned = hdb["child_pop_0_9"].sum()
available = child_pop["child_pop"].sum()
unassigned = available - assigned
print("\n[1] Floor area source for dwelling units")
print(f"    Per-block (from resale transactions) : {unit_stats['per_block_units']:>9,.0f} units  "
      f"({unit_stats['per_block_units'] / unit_stats['total_units'] * 100:.1f}%)")
print(f"    National median fallback             : {unit_stats['fallback_units']:>9,.0f} units  "
      f"({unit_stats['fallback_units'] / unit_stats['total_units'] * 100:.1f}%)")
print("\n[2] Child population assigned (ages 0-9)")
print(f"    Assigned to HDB blocks : {assigned:>9,.0f}  ({assigned / available * 100:.1f}%)")
print(f"    Unassigned (private housing, expected) : {unassigned:>9,.0f}  ({unassigned / available * 100:.1f}%)")

## 6. Write the layer

In [ ]:
keep = (
    ["geometry", "join_key", "subzone", "total_dwelling_units", "child_pop_0_9"]
    + [f"units_{b}" for b in FA_BUCKETS]
)
keep = [c for c in keep if c in hdb.columns]
hdb[keep].to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print(f"Saved -> {OUT_GPKG.relative_to(ROOT)}  (layer: {OUT_LAYER}, {len(hdb):,} blocks)")
print(hdb[["join_key", "subzone", "total_dwelling_units", "child_pop_0_9"]].dropna().head(10).to_string(index=False))